# Task 2: Mobil
## Keyword Search (BM25 & TF-IDF) dan Semantic Search (Vector Database)

Pada notebook ini, kita akan membuat sistem pencarian mobil dengan dua pendekatan:
1. *Keyword Search*: BM25 dan TF-IDF untuk pencarian berbasis kata kunci
2. *Semantic Search*: Menggunakan embeddings dan vector database (Pinecone & ChromaDB)

In [1]:
!pip install rank_bm25 scikit-learn pandas numpy chromadb pinecone-client sentence-transformers


In [2]:
import pandas as pd
import numpy as np
import os
import warnings
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

# Keyword Search
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Semantic Search
from sentence_transformers import SentenceTransformer
import chromadb
from pinecone import Pinecone, ServerlessSpec

load_dotenv()
print("Libraries imported successfully.")

Libraries imported successfully.


Load dataset

In [ ]:

df = pd.read_csv('data_mobil.csv')

df['price_formatted'] = df['price_idr'].apply(lambda x: f"Rp {x:,.0f}".replace(',', '.'))

# Create searchable text (kombinasi brand, model, specs, dan description)
df['searchable_text'] = (df['brand'].fillna('') + ' ' + 
                         df['model'].fillna('') + ' ' + 
                         df['specs'].fillna('') + ' ' + 
                         df['description'].fillna(''))

print(f"Dataset shape: {df.shape}")
df.head(3)

Dataset shape: (20, 9)


,id,brand,model,year,price_idr,specs,description,price_formatted,searchable_text
0,1,Toyota,Avanza,2024,250000000,"1.5L, 4-cylinder, manual/automatic, 7-seater",Mobil keluarga 7 penumpang dengan konsumsi bah...,Rp 250.000.000,"Toyota Avanza 1.5L, 4-cylinder, manual/automat..."
1,2,Honda,Brio,2023,180000000,"1.2L engine, CVT, compact hatchback",City car kecil yang cocok untuk penggunaan har...,Rp 180.000.000,"Honda Brio 1.2L engine, CVT, compact hatchback..."
2,3,Mitsubishi,Xpander,2024,280000000,"1.5L engine, automatic, spacious interior",Mobil MPV dengan kabin luas dan nyaman untuk p...,Rp 280.000.000,"Mitsubishi Xpander 1.5L engine, automatic, spa..."


BM25 Setup & Function

In [4]:
# Prepare corpus
corpus = df['searchable_text'].tolist()
tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

def search_bm25(query, top_k=3):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        if scores[idx] > 0: # Only append if there's a match
            row = df.iloc[idx]
            results.append({
                'brand_model': f"{row['brand']} {row['model']}",
                'year': row['year'],
                'price': row['price_formatted'],
                'specs': row['specs'],
                'score': scores[idx]
            })
    return results

TF-IDF Setup & Function

In [5]:
tfidf_vectorizer = TfidfVectorizer(stop_words=None, ngram_range=(1, 2))
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

def search_tfidf(query, top_k=3):
    query_vec = tfidf_vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        if similarities[idx] > 0:
            row = df.iloc[idx]
            results.append({
                'brand_model': f"{row['brand']} {row['model']}",
                'year': row['year'],
                'price': row['price_formatted'],
                'specs': row['specs'],
                'score': similarities[idx]
            })
    return results

In [6]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("Embedding model loaded.")
print(f"Dimension: {model.get_sentence_embedding_dimension()}")

# Generate embeddings for the dataset
embeddings = model.encode(corpus, show_progress_bar=True)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7193.42it/s]


Embedding model loaded.
Dimension: 384


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.30it/s]


ChromaDB Setup & Function (Local Vector DB)

In [7]:
# Initialize Persistent ChromaDB
chroma_client = chromadb.PersistentClient(path="./chroma_mobil")

# Reset collection if exists
try:
    chroma_client.delete_collection(name="mobil_collection")
except:
    pass

collection = chroma_client.create_collection(
    name="mobil_collection",
    metadata={"hnsw:space": "cosine"}
)

# Insert data
collection.add(
    embeddings=embeddings.tolist(),
    documents=corpus,
    metadatas=[{
        'brand_model': f"{row['brand']} {row['model']}",
        'year': str(row['year']),
        'price': row['price_formatted'],
        'specs': row['specs']
    } for _, row in df.iterrows()],
    ids=[str(row['id']) for _, row in df.iterrows()]
)
print("Data inserted to ChromaDB.")

def search_chroma(query, top_k=3):
    query_embedding = model.encode([query])
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k
    )
    
    formatted_results = []
    for i in range(len(results['ids'][0])):
        meta = results['metadatas'][0][i]
        formatted_results.append({
            'brand_model': meta['brand_model'],
            'year': meta['year'],
            'price': meta['price'],
            'specs': meta['specs'],
            'distance': results['distances'][0][i]
        })
    return formatted_results

Data inserted to ChromaDB.


 Pinecone Setup & Function (Cloud Vector DB)

In [13]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec

In [ ]:
load_dotenv(override=True)

PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')

if PINECONE_API_KEY:
    print(f"✓ API Key terbaca: {PINECONE_API_KEY[:5]}...{PINECONE_API_KEY[-3:]}")
    
    try:
        pc = Pinecone(api_key=PINECONE_API_KEY)
        index_name = 'mobil-search'
        
        if index_name not in pc.list_indexes().names():
            print(f"Creating index '{index_name}'...")
            pc.create_index(
                name=index_name,
                dimension=384, 
                metric='cosine',
                spec=ServerlessSpec(
                    cloud='aws',
                    region='us-east-1' 
                )
            )
            print(f"✓ Index '{index_name}' created.")
        else:
            print(f"✓ Index '{index_name}' already exists.")
            
        pinecone_index = pc.Index(index_name)
        
        print("Mempersiapkan data untuk diupload...")
        # Upload data
        vectors = []
        for i, row in df.iterrows():
            vectors.append({
                'id': str(row['id']),
                'values': embeddings[i].tolist(),
                'metadata': {
                    'brand_model': f"{row['brand']} {row['model']}",
                    'year': str(row['year']),
                    'price': row['price_formatted'],
                    'specs': row['specs']
                }
            })
            
    
        pinecone_index.upsert(vectors=vectors)
        print(f"✓ Berhasil mengunggah {len(vectors)} records ke Pinecone!")
        
    except Exception as e:
        print(f"⚠ Terjadi error saat proses Pinecone: {e}")
        
else:
    print("⚠ PINECONE_API_KEY masih kosong! Pastikan file .env sudah benar dan berada di folder yang sama dengan notebook.")


✓ API Key terbaca: pcsk_...W7A
✓ Index 'mobil-search' already exists.
Mempersiapkan data untuk diupload...
✓ Berhasil mengunggah 20 records ke Pinecone!


testing

In [16]:
test_queries = [
    "Mobil yang cocok untuk keluarga",
    "Mobil Listrik",
    "Mobil Autopilot",
    "Mobil hemat bahan bakar",
    "Mobil murah dibawah 250 juta dan keluaran tahun 2023"
]

print("=== CHROMA DB SEMANTIC SEARCH TEST ===\n")
for q in test_queries:
    print(f"Query: '{q}'")
    results = search_chroma(q, top_k=2)
    for res in results:
        print(f" - {res['brand_model']} ({res['year']}) | {res['price']} | Dist: {res['distance']:.4f}")
    print("-" * 50)

=== CHROMA DB SEMANTIC SEARCH TEST ===

Query: 'Mobil yang cocok untuk keluarga'
 - Daihatsu Terios (2023) | Rp 260.000.000 | Dist: 0.2973
 - Toyota Avanza (2024) | Rp 250.000.000 | Dist: 0.3055
--------------------------------------------------
Query: 'Mobil Listrik'
 - Wuling Air EV (2024) | Rp 300.000.000 | Dist: 0.6824
 - Hyundai Ioniq 5 (2024) | Rp 750.000.000 | Dist: 0.7094
--------------------------------------------------
Query: 'Mobil Autopilot'
 - Tesla Model 3 (2024) | Rp 900.000.000 | Dist: 0.4933
 - Mitsubishi Xpander (2024) | Rp 280.000.000 | Dist: 0.7585
--------------------------------------------------
Query: 'Mobil hemat bahan bakar'
 - Toyota Avanza (2024) | Rp 250.000.000 | Dist: 0.3673
 - Daihatsu Ayla (2023) | Rp 150.000.000 | Dist: 0.3997
--------------------------------------------------
Query: 'Mobil murah dibawah 250 juta dan keluaran tahun 2023'
 - Toyota Feruza (2022) | Rp 200.000.000 | Dist: 0.4720
 - Daihatsu Ayla (2023) | Rp 150.000.000 | Dist: 0.5245
---